In [0]:
%sql

-- GeoCEP - Data Quality das duas APIs
-- Verifica schema, nulos, duplicidade e intervalos

CREATE OR REPLACE TABLE geocep.default.dq_cep
USING DELTA
AS

WITH fontes AS (

    -- Unifica as duas fontes para a auditoria
    SELECT * FROM geocep.default.ref_viacep
    UNION ALL
    SELECT * FROM geocep.default.ref_brasilapi

),

metricas AS (

    SELECT
        fonte,
        CURRENT_TIMESTAMP() AS data_auditoria,

        -- Volume e unicidade
        COUNT(*) AS total_registros,
        COUNT(DISTINCT cep) AS ceps_distintos,

        COUNT(*) - COUNT(DISTINCT cep) AS
            registros_excedentes_ou_sem_cep,

        -- Verificacao de nulos
        COUNT_IF(cep IS NULL) AS cep_nulo,
        COUNT_IF(data_consulta IS NULL) AS data_nula,
        COUNT_IF(json_original IS NULL) AS json_nulo,

        ROUND(
            100.0 * COUNT_IF(json_original IS NULL)
            / NULLIF(COUNT(*), 0), 2
        ) AS null_ratio_json,

        -- Teste de intervalo/formato do CEP
        COUNT_IF(
            cep IS NULL OR NOT (cep RLIKE '^[0-9]{8}$')
        ) AS cep_invalido,

        -- Teste de intervalo das datas
        COUNT_IF(
            data_consulta IS NULL
            OR data_consulta > CURRENT_TIMESTAMP()
            OR data_consulta < '2020-01-01'
        ) AS data_invalida,

        -- Qualidade das respostas HTTP
        COUNT_IF(status = 'OK') AS consultas_ok,

        COUNT_IF(
            status <> 'OK' OR status IS NULL
        ) AS consultas_com_problema,

        -- Verificacao dos tipos do schema
        COUNT_IF(
            TYPEOF(cep) <> 'string'
            OR TYPEOF(data_consulta) <> 'timestamp'
            OR TYPEOF(json_original) <> 'string'
        ) AS erros_schema

    FROM fontes
    GROUP BY fonte
)

SELECT
    *,
    CASE
        WHEN total_registros = 0
          OR registros_excedentes_ou_sem_cep > 0
          OR cep_invalido > 0
          OR data_invalida > 0
          OR json_nulo > 0
          OR erros_schema > 0
        THEN 'REPROVADO'
        ELSE 'APROVADO'
    END AS status_dq

FROM metricas;


In [0]:
%sql
SELECT *
FROM geocep.default.dq_cep;

In [0]:
import json

# Busca os resultados da auditoria
dados = spark.table("geocep.default.dq_cep")

# Salva o relatório JSON no Volume
caminho = "/Volumes/geocep/default/geocep/quality"

dbutils.fs.mkdirs(caminho)

relatorio = [r.asDict() for r in dados.collect()]

# Converte timestamps para texto na exportação
texto = json.dumps(
    relatorio,
    indent=2,
    ensure_ascii=False,
    default=str
)

dbutils.fs.put(
    f"{caminho}/relatorio_dq.json",
    texto,
    overwrite=True
)

print("Relatório de Data Quality gerado!")

In [0]:
import json

caminho = "/Volumes/geocep/default/geocep/quality/relatorio_dq.json"

with open(caminho, "r", encoding="utf-8") as arquivo:
    relatorio = json.load(arquivo)

print(json.dumps(relatorio, indent=4, ensure_ascii=False))